# LAIQMS Capstone: LLM-Grounded Alpha Research and Risk System

This is the full bootcamp capstone in a natural financial-model-building order:

1. Quant data, returns, labels, options, and baseline risk.
2. Feature engine using data structures and system-design discipline.
3. Walk-forward ML/AI model with honest validation.
4. LLM/RAG research layer grounded in retrieved evidence and market data.
5. Portfolio construction, execution costs, option overlay, and risk report.
6. AWS/system design package with security, monitoring, cost, and final-defense prompts.

This is the capstone project from Columbia MAFN's LAIQMS bootcamp — a program preparing incoming students for AI/ML/LLM/quant/systems-design interviews. See the repository README for context.


## LAIQMS Coverage Map

| Module | LAIQMS | Main subsections |
|---|---|---|
| S1 Data, Returns, Labels | Q, M | Calculus, Linear Algebra, Probability, Stochastic, Complex, Instruments, Portfolio, Black-Scholes, Monte Carlo, Time-Series Validation |
| S2 Feature Engine + DSA | S, Q | Arrays, Strings, Linked Lists, Stacks, Queues, Sorting, Binary Search, Recursion, Trees, Graphs, Heaps, Monotonic Stacks, Dynamic Programming, Greedy/Cache, System Estimation |
| S3 ML + AI Validation | M, AI, Q | Time-Series CV, Supervised Learning, Trees/Ensembles, Unsupervised Learning, Metrics, Neural Networks, Deep Architectures, Embeddings, Optimization, Calibration, Deployment |
| S4 LLM Research Layer | L, AI, Q | Tokenization, Embeddings, Softmax, Attention, Transformer Blocks, Backpropagation, Scaling/FLOPs/Memory, Evaluation, RAG |
| S5 Portfolio, Execution, Risk | Q, M, S | Portfolio Theory, Instruments, Black-Scholes/Greeks, Monte Carlo, Model Metrics, Stream/Queue Controls, Trading System Design |
| S6 AWS + Final Defense | S, AWS, L, AI, Q, M | AWS Cloud, IAM, Compute, Storage, Networking, Monitoring, Billing, Applied ML/Trading System Design |


# S1 - Data, Returns, Labels

Build the market data contract, leak-free return labels, PCA risk model, option-pricing layer, and baseline portfolio math.


## Mapped LAIQMS Subsections

- Q: Calculus & Differential Equations
- Q: Linear Algebra
- Q: Probability & Statistics
- Q: Stochastic Processes
- Q: Complex Numbers
- Q: Financial Instruments & Derivatives
- Q: Portfolio Theory, Factors & Risk
- Q: Black–Scholes
- Q: Monte Carlo & Numerical Methods
- M: Time-Series Validation & Leakage Control


## S1 - Data, Returns, Labels

**Natural financial-model order:** before ML, LLMs, or AWS, the system needs a clean market data contract.

**LAIQMS mapping**
- Q / Calculus: log returns as discrete derivatives of log price.
- Q / Linear Algebra: covariance, eigenvalues, PCA, Cholesky.
- Q / Probability & Statistics: moments, quantiles, VaR, expected shortfall.
- Q / Stochastic Processes: regime-switching GBM-like simulation.
- Q / Complex Numbers: FFT identifies cyclic structure in returns.
- Q / Financial Instruments: equity, option metadata, risk-free curve proxy.
- Q / Portfolio Theory: covariance shrinkage, beta, Sharpe, risk budget.
- Q / Black-Scholes: option price and Greeks.
- Q / Monte Carlo: simulated terminal distributions and option checks.
- M / Time-Series Validation: point-in-time labels only use future returns after the feature date.


In [1]:
import json
import math
import os
import heapq
import bisect
from collections import deque, defaultdict
from pathlib import Path

import numpy as np
import pandas as pd

from scipy.stats import norm
from scipy.optimize import brentq
from sklearn.decomposition import PCA

RNG_SEED = 113353
rng = np.random.default_rng(RNG_SEED)

ARTIFACTS = Path("artifacts")
ARTIFACTS.mkdir(exist_ok=True)

TICKERS = ["AAPL", "MSFT", "JPM", "GS", "XOM", "UNH", "CAT", "GE", "IBM", "DIS", "KO", "WMT"]
N_DAYS = 756
DATES = pd.bdate_range("2023-01-03", periods=N_DAYS)
RISK_FREE = 0.043

# --- Market-data source -----------------------------------------------------
# Point MARKET_DATA_BASE_URL at a CORS-enabled S3/CloudFront prefix that holds
# per-symbol CSVs (capstone/market-data/<SYMBOL>.csv with columns
# date,open,high,low,close,volume). A backend job downloads these from Yahoo
# Finance and uploads them to S3, so the same URL works on your laptop AND in
# the in-browser JupyterLite IDE. Leave it as "" to use the reproducible
# synthetic universe (the offline / in-browser default).
MARKET_DATA_BASE_URL = "https://mafn-bootcamp-2026.s3.us-east-1.amazonaws.com"  # public Yahoo->S3 CSVs; "" -> synthetic
ASSET_CLASS_ETFS = {"Equities": "SPY", "Bonds": "AGG", "Commodities": "DBC", "Cash": "BIL"}
ASSET_CLASSES = list(ASSET_CLASS_ETFS)

def annualize_return(daily_mean):
    return (1.0 + daily_mean) ** 252 - 1.0

def annualize_vol(daily_std):
    return daily_std * np.sqrt(252)

def max_drawdown(series):
    wealth = pd.Series(series).astype(float)
    peak = wealth.cummax()
    dd = wealth / peak - 1.0
    return float(dd.min())

def black_scholes_call(S, K, T, r, sigma):
    sigma = max(float(sigma), 1e-8)
    T = max(float(T), 1e-8)
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    price = S * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)
    delta = norm.cdf(d1)
    gamma = norm.pdf(d1) / (S * sigma * np.sqrt(T))
    vega = S * norm.pdf(d1) * np.sqrt(T)
    theta = -(S * norm.pdf(d1) * sigma) / (2 * np.sqrt(T)) - r * K * np.exp(-r * T) * norm.cdf(d2)
    return {"price": float(price), "delta": float(delta), "gamma": float(gamma), "vega": float(vega), "theta": float(theta)}

def implied_vol_call(price, S, K, T, r):
    def f(sig):
        return black_scholes_call(S, K, T, r, sig)["price"] - price
    try:
        return float(brentq(f, 1e-4, 4.0, maxiter=100))
    except ValueError:
        return np.nan

print("Setup complete. Seed:", RNG_SEED)


Setup complete. Seed: 113353


In [2]:
# Market-data contract: REAL (Yahoo -> public S3, CORS-enabled) when
# MARKET_DATA_BASE_URL is set, otherwise a reproducible synthetic factor
# universe. Both paths yield the SAME downstream contract: prices, returns,
# volumes, regime. The synthetic path keeps the capstone fully offline and
# in-browser friendly (Pyodide/JupyterLite cannot reach Yahoo directly).
n = len(TICKERS)

def synthesize_universe():
    # Latent macro/sector/liquidity/volatility factors with regime switching.
    k = 5
    factor_names = ["market", "rates", "energy", "quality", "liquidity"]
    regime = np.zeros(N_DAYS, dtype=int)
    p_stay = {0: 0.965, 1: 0.92, 2: 0.88}
    for t in range(1, N_DAYS):
        if rng.random() < p_stay[regime[t - 1]]:
            regime[t] = regime[t - 1]
        else:
            regime[t] = rng.choice([0, 1, 2], p=[0.72, 0.18, 0.10])
    regime_mu = np.array([0.00035, -0.00015, -0.00055])
    regime_vol = np.array([0.0085, 0.014, 0.024])
    factor_cov = np.array([
        [1.00, -0.22, 0.15, 0.30, -0.35],
        [-0.22, 1.00, -0.08, -0.10, 0.25],
        [0.15, -0.08, 1.00, 0.04, -0.10],
        [0.30, -0.10, 0.04, 1.00, -0.28],
        [-0.35, 0.25, -0.10, -0.28, 1.00],
    ])
    Lf = np.linalg.cholesky(factor_cov)
    raw = rng.standard_normal((N_DAYS, k)) @ Lf.T
    factor_rets = pd.DataFrame(raw * regime_vol[regime, None] + regime_mu[regime, None] / k, index=DATES, columns=factor_names)
    exposures = pd.DataFrame(rng.normal(0.0, 0.55, size=(n, k)), index=TICKERS, columns=factor_names)
    exposures["market"] = rng.normal(1.0, 0.20, n)
    exposures["quality"] = rng.normal(0.25, 0.45, n)
    idio = rng.normal(0, 0.0065, size=(N_DAYS, n))
    drift = rng.normal(0.00018, 0.00008, n)
    rets = factor_rets.values @ exposures.T.values + drift + idio
    rets = pd.DataFrame(rets, index=DATES, columns=TICKERS).clip(-0.12, 0.12)
    px = 100 * np.exp(rets.cumsum())
    vol = pd.DataFrame(
        rng.lognormal(mean=14.4, sigma=0.33, size=(N_DAYS, n)) * (1.0 + np.abs(rets.values) * 18),
        index=DATES, columns=TICKERS,
    )
    return px, rets, vol, pd.Series(regime, index=DATES, name="regime")

def regime_from_returns(rets):
    # Derive a 3-state risk regime from realized cross-sectional volatility terciles.
    rv = rets.std(axis=1).rolling(21).mean().bfill()
    hi, crisis = rv.quantile(0.60), rv.quantile(0.85)
    state = pd.Series(0, index=rets.index, name="regime")
    state[rv > hi] = 1
    state[rv > crisis] = 2
    return state

def load_csv_series(symbol, column):
    df = pd.read_csv(f"{MARKET_DATA_BASE_URL}/{symbol}.csv")
    df.columns = [c.lower() for c in df.columns]
    df["date"] = pd.to_datetime(df["date"])
    return df.set_index("date")[column].sort_index()

def load_equity_panel():
    # Real Yahoo -> S3 prices/volumes when configured; synthetic fallback on any failure.
    if MARKET_DATA_BASE_URL:
        try:
            close, vol = {}, {}
            for t in TICKERS:
                close[t] = load_csv_series(t, "close")
                vol[t] = load_csv_series(t, "volume")
            px = pd.DataFrame(close).dropna(how="all").sort_index()
            vl = pd.DataFrame(vol).reindex(px.index)
            if len(px) >= 252 and px.shape[1] == len(TICKERS):
                rets = px.pct_change().clip(-0.25, 0.25)
                print(f"DATA SOURCE: real Yahoo->S3 ({px.shape[0]} days x {px.shape[1]} tickers)")
                return px, rets, vl, regime_from_returns(rets), "yahoo_s3"
            print("Real data incomplete; using synthetic universe.")
        except Exception as exc:
            print(f"Real-data load failed ({exc!r}); using synthetic universe.")
    px, rets, vl, reg = synthesize_universe()
    print("DATA SOURCE: synthetic (reproducible, offline/in-browser safe)")
    return px, rets, vl, reg, "synthetic"

prices, returns, volumes, regime_series, DATA_SOURCE = load_equity_panel()

def load_asset_classes():
    # Multi-asset class returns (Equities/Bonds/Commodities/Cash) for the frontier.
    if MARKET_DATA_BASE_URL:
        try:
            cols = {cls: load_csv_series(etf, "close").pct_change() for cls, etf in ASSET_CLASS_ETFS.items()}
            ac = pd.DataFrame(cols).reindex(returns.index).dropna()
            if len(ac) >= 252:
                return ac, "yahoo_s3"
        except Exception as exc:
            print(f"Asset-class real load failed ({exc!r}); synthesizing.")
    # Realistic ETF-like daily stats (Equities/Bonds/Commodities/Cash).
    mu = np.array([0.000317, 0.000119, 0.000198, 0.000171])
    sd = np.array([0.01008, 0.00315, 0.00945, 0.00060])
    corr_ac = np.array([
        [1.00, -0.30, 0.25, 0.00],
        [-0.30, 1.00, -0.10, 0.05],
        [0.25, -0.10, 1.00, 0.00],
        [0.00, 0.05, 0.00, 1.00],
    ])
    z = rng.standard_normal((len(returns), 4)) @ np.linalg.cholesky(corr_ac).T
    ac = pd.DataFrame(z * sd + mu, index=returns.index, columns=ASSET_CLASSES)
    return ac, "synthetic"

asset_class_returns, ASSET_CLASS_SOURCE = load_asset_classes()

# Leak-free 5-day-ahead cross-sectional label (computed AFTER the source is final).
fwd5 = returns.shift(-5).rolling(5).sum().shift(-4)
labels = (fwd5 > fwd5.median(axis=1).values[:, None]).astype(int)
labels = labels.iloc[:-10]

DATA = {
    "prices": prices, "returns": returns, "volumes": volumes,
    "regime": regime_series, "asset_classes": asset_class_returns, "source": DATA_SOURCE,
}
print(prices.tail(2).round(2))
print("Returns shape:", returns.shape, "Labels shape:", labels.shape, "Asset classes:", list(asset_class_returns.columns))


DATA SOURCE: real Yahoo->S3 (751 days x 12 tickers)


              AAPL   MSFT     JPM       GS     XOM     UNH     CAT      GE  \
date                                                                         
2026-07-24  333.02  381.7  353.21  1061.23  156.94  420.74  888.73  353.73   
2026-07-27  336.91  389.1  356.20  1048.23  154.77  417.64  873.28  361.61   

               IBM    DIS     KO     WMT  
date                                      
2026-07-24  214.19  94.85  82.25  109.47  
2026-07-27  216.28  96.65  84.07  111.74  
Returns shape: (751, 12) Labels shape: (741, 12) Asset classes: ['Equities', 'Bonds', 'Commodities', 'Cash']


In [3]:
# Quant audit: moments, PCA, FFT, risk, portfolio baseline, and option pricing.
ret = DATA["returns"].dropna()
mu_daily = ret.mean()
cov_daily = ret.cov()
cov_ann = cov_daily * 252
vol_ann = ret.std() * np.sqrt(252)
corr = ret.corr()

eigvals, eigvecs = np.linalg.eigh(cov_ann.values)
order = eigvals.argsort()[::-1]
eigvals = eigvals[order]
eigvecs = eigvecs[:, order]
pca = PCA(n_components=4, random_state=RNG_SEED).fit(ret.fillna(0.0))

# Correlation risk (consumed, not just computed): average pairwise correlation
# and the absorption ratio (share of variance in the top eigenvector) flag crowding.
triu = np.triu_indices(len(TICKERS), k=1)
avg_pairwise_corr = float(corr.values[triu].mean())
absorption_ratio = float(eigvals[0] / eigvals.sum())

# Complex numbers: FFT on market return proxy.
market_proxy = ret.mean(axis=1).values
fft = np.fft.rfft(market_proxy - market_proxy.mean())
freqs = np.fft.rfftfreq(len(market_proxy), d=1)
dominant = int(np.argmax(np.abs(fft[1:])) + 1)
cycle_days = float(1.0 / freqs[dominant]) if freqs[dominant] else np.inf

# Mean-variance baseline with ridge-stabilized inverse covariance.
mu_ann = mu_daily * 252
shrink = 0.12
diag = np.diag(np.diag(cov_ann.values))
sigma_shrunk = (1 - shrink) * cov_ann.values + shrink * diag
inv_sigma = np.linalg.pinv(sigma_shrunk)
raw_w = inv_sigma @ (mu_ann.values - RISK_FREE)
w = raw_w / np.sum(np.abs(raw_w))
weights = pd.Series(w, index=TICKERS, name="baseline_weight")

port_ret = (ret @ weights).rename("baseline_portfolio")
wealth = (1 + port_ret).cumprod()
var95 = float(np.quantile(port_ret, 0.05))
es95 = float(port_ret[port_ret <= var95].mean())
sharpe = float((port_ret.mean() * 252 - RISK_FREE) / (port_ret.std() * np.sqrt(252)))

S0 = float(prices.iloc[-1]["AAPL"])
sigma_aapl = float(vol_ann["AAPL"])
opt = black_scholes_call(S0, S0 * 1.02, 30 / 365, RISK_FREE, sigma_aapl)
iv_check = implied_vol_call(opt["price"], S0, S0 * 1.02, 30 / 365, RISK_FREE)

# Monte Carlo sanity check for the same option.
z = rng.standard_normal(25000)
ST = S0 * np.exp((RISK_FREE - 0.5 * sigma_aapl**2) * (30 / 365) + sigma_aapl * np.sqrt(30 / 365) * z)
mc_price = float(np.exp(-RISK_FREE * (30 / 365)) * np.maximum(ST - S0 * 1.02, 0).mean())

# Portfolio Monte Carlo: simulate forward portfolio returns from the estimated
# daily mean/cov (multivariate normal) and summarize the terminal-wealth distribution.
mc_horizon, mc_paths = 21, 20000
cov_daily_psd = sigma_shrunk / 252.0 + 1e-12 * np.eye(len(TICKERS))
chol_daily = np.linalg.cholesky(cov_daily_psd)
sim = rng.standard_normal((mc_paths, mc_horizon, len(TICKERS))) @ chol_daily.T + mu_daily.values
sim_terminal = (1.0 + (sim @ weights.values)).prod(axis=1)
portfolio_mc = {
    "horizon_days": mc_horizon,
    "n_paths": mc_paths,
    "median_terminal_wealth": float(np.median(sim_terminal)),
    "p05_terminal_wealth": float(np.quantile(sim_terminal, 0.05)),
    "p95_terminal_wealth": float(np.quantile(sim_terminal, 0.95)),
    "mc_var_95": float(np.quantile(sim_terminal - 1.0, 0.05)),
    "prob_loss": float((sim_terminal < 1.0).mean()),
}

s1_report = {
    "laiqms_modules": ["Q", "M"],
    "data_source": DATA["source"],
    "asset_class_source": ASSET_CLASS_SOURCE,
    "dominant_fft_cycle_days": cycle_days,
    "pca_explained_variance": pca.explained_variance_ratio_.round(4).tolist(),
    "portfolio_sharpe": sharpe,
    "daily_var_95": var95,
    "daily_expected_shortfall_95": es95,
    "max_drawdown": max_drawdown(wealth),
    "correlation_risk": {"avg_pairwise_corr": avg_pairwise_corr, "absorption_ratio": absorption_ratio},
    "bs_call": opt,
    "implied_vol_check": iv_check,
    "mc_call_price": mc_price,
    "portfolio_monte_carlo": portfolio_mc,
    "asset_class_ann_return": (asset_class_returns.mean() * 252).round(4).to_dict(),
    "asset_class_ann_vol": (asset_class_returns.std() * np.sqrt(252)).round(4).to_dict(),
    "weights": weights.round(4).to_dict(),
    "wealth_curve": {"dates": [str(d.date()) for d in wealth.index], "wealth": wealth.round(5).tolist()},
}
(ARTIFACTS / "s1_quant_data_contract.json").write_text(json.dumps(s1_report, indent=2))
print(json.dumps({k: s1_report[k] for k in ["data_source", "portfolio_sharpe", "daily_var_95", "mc_call_price", "correlation_risk", "portfolio_monte_carlo"]}, indent=2))


{
  "data_source": "yahoo_s3",
  "portfolio_sharpe": 1.8674265268963304,
  "daily_var_95": -0.009126112579612402,
  "mc_call_price": 7.787954082655983,
  "correlation_risk": {
    "avg_pairwise_corr": 0.18293177769123742,
    "absorption_ratio": 0.2983933773571652
  },
  "portfolio_monte_carlo": {
    "horizon_days": 21,
    "n_paths": 20000,
    "median_terminal_wealth": 1.0200186486489122,
    "p05_terminal_wealth": 0.9698713047046457,
    "p95_terminal_wealth": 1.0734177416498394,
    "mc_var_95": -0.03012869529535435,
    "prob_loss": 0.26165
  }
}


# S2 - Feature Engine + Data Structures

Convert quant concepts into point-in-time model features using every core coding-interview pattern and a pipeline DAG.


## Mapped LAIQMS Subsections

- S: Arrays
- S: Strings
- S: Linked Lists
- S: Stacks
- S: Queues
- S: Sorting
- S: Binary Search
- S: Recursion and Backtracking
- S: Trees
- S: Graphs
- S: Heaps and Priority Queues
- S: Monotonic Stacks
- S: Dynamic Programming
- S: Ad Hoc, Greedy and Cache Design
- S: System Design Fundamentals & Estimation


## S2 - Feature Engine + Data Structures

**Natural order:** once the quant data contract exists, build features. This is where coding-interview patterns become finance infrastructure.

**LAIQMS mapping**
- S / Arrays: vectorized returns, rolling windows, feature matrices.
- S / Strings: ticker, event, and feature-name parsing.
- S / Linked Lists: sparse ordered event chains and pointer reasoning.
- S / Stacks: nested validation gates and rollback reasoning.
- S / Queues: rolling-window updates and event buffering.
- S / Sorting: cross-sectional ranks and stable feature ordering.
- S / Binary Search: locate event dates and rebalance boundaries.
- S / Recursion and Backtracking: dependency walks and invalid-branch pruning.
- S / Trees: hierarchical feature registry.
- S / Graphs: pipeline dependency DAG and topological order.
- S / Heaps: top-k momentum candidates.
- S / Monotonic Stacks: rolling drawdown and maximum tracking.
- S / Dynamic Programming: transaction-cost-aware position smoothing.
- S / Ad Hoc, Greedy and Cache Design: invariants, cache discipline, and greedy pruning.
- S / System Design Fundamentals: model-table sizing, latency, and storage math.


In [4]:
def rolling_zscore(df, window):
    mean = df.rolling(window).mean()
    std = df.rolling(window).std().replace(0, np.nan)
    return (df - mean) / std

def deque_rolling_mean(values, window):
    q, total, out = deque(), 0.0, []
    for x in values:
        q.append(float(x)); total += float(x)
        if len(q) > window:
            total -= q.popleft()
        out.append(total / len(q))
    return np.array(out)

def rolling_max_drawdown(values, window=63):
    out = []
    for i in range(len(values)):
        w = np.asarray(values[max(0, i - window + 1): i + 1], dtype=float)
        if len(w) < 2:
            out.append(0.0)
        else:
            out.append(max_drawdown(pd.Series(w / w[0])))
    return np.array(out)

def top_k_momentum(momentum_row, k=4):
    heap = []
    for ticker, val in momentum_row.dropna().items():
        heapq.heappush(heap, (float(val), ticker))
        if len(heap) > k:
            heapq.heappop(heap)
    return sorted([(t, v) for v, t in heap], key=lambda x: -x[1])

def topo_sort(graph):
    indeg = defaultdict(int)
    for node, deps in graph.items():
        indeg[node] += 0
        for dep in deps:
            indeg[dep] += 0
            indeg[node] += 1
    q = deque([n for n, d in indeg.items() if d == 0])
    order = []
    while q:
        n = q.popleft()
        order.append(n)
        for child, deps in graph.items():
            if n in deps:
                indeg[child] -= 1
                if indeg[child] == 0:
                    q.append(child)
    if len(order) != len(indeg):
        raise ValueError("Cycle in feature DAG")
    return order


In [5]:
ret = DATA["returns"]
prices = DATA["prices"]
volumes = DATA["volumes"]

mom_5 = prices.pct_change(5)
mom_21 = prices.pct_change(21)
vol_21 = ret.rolling(21).std() * np.sqrt(252)
vol_z = rolling_zscore(vol_21, 63)
dollar_vol_z = rolling_zscore(prices * volumes, 63)

beta_63 = pd.DataFrame(index=ret.index, columns=TICKERS, dtype=float)
market = ret.mean(axis=1)
for t in TICKERS:
    cov = ret[t].rolling(63).cov(market)
    var = market.rolling(63).var()
    beta_63[t] = cov / var

drawdown_63 = pd.DataFrame({t: rolling_max_drawdown(prices[t].values, 63) for t in TICKERS}, index=prices.index)
top_momentum_last = top_k_momentum(mom_21.iloc[-1], k=4)
rebalance_dates = list(prices.index[::21])
locate_idx = bisect.bisect_left(rebalance_dates, prices.index[250])

feature_panels = {
    "mom_5": mom_5,
    "mom_21": mom_21,
    "vol_21": vol_21,
    "vol_z": vol_z,
    "dollar_vol_z": dollar_vol_z,
    "beta_63": beta_63,
    "drawdown_63": drawdown_63,
}
rows = []
for name, panel in feature_panels.items():
    tmp = panel.stack().rename(name).reset_index()
    tmp.columns = ["date", "ticker", name]
    rows.append(tmp)
features = rows[0]
for r in rows[1:]:
    features = features.merge(r, on=["date", "ticker"], how="outer")

# Join the leak-free label from S1.
labels_long = labels.stack().rename("target").reset_index()
labels_long.columns = ["date", "ticker", "target"]
model_table = features.merge(labels_long, on=["date", "ticker"], how="inner").dropna()

FEATURE_COLUMNS = ["mom_5", "mom_21", "vol_21", "vol_z", "dollar_vol_z", "beta_63", "drawdown_63"]
PIPELINE_DAG = {
    "raw_prices": [],
    "returns": ["raw_prices"],
    "labels": ["returns"],
    "rolling_features": ["returns", "raw_prices"],
    "risk_features": ["returns"],
    "model_table": ["labels", "rolling_features", "risk_features"],
    "walk_forward_model": ["model_table"],
    "portfolio": ["walk_forward_model", "risk_features"],
}
dag_order = topo_sort(PIPELINE_DAG)

S2 = {
    "laiqms_modules": ["S", "Q"],
    "feature_columns": FEATURE_COLUMNS,
    "model_table_rows": int(len(model_table)),
    "top_momentum_last": top_momentum_last,
    "rebalance_date_example_index": int(locate_idx),
    "pipeline_topological_order": dag_order,
    "complexities": {
        "rolling_features_vectorized": "roughly O(T*N) per feature",
        "top_k_heap": "O(N log k)",
        "date_binary_search": "O(log T)",
        "dag_topological_sort": "O(V+E)",
    },
}
(ARTIFACTS / "s2_feature_catalog.json").write_text(json.dumps(S2, indent=2))
print(model_table.tail(3).round(4))
print(json.dumps({k: S2[k] for k in ["model_table_rows", "pipeline_topological_order"]}, indent=2))


           date ticker   mom_5  mom_21  vol_21   vol_z  dollar_vol_z  beta_63  \
8889 2026-07-13    UNH  0.0266  0.0531  0.2483 -0.8847       -0.9364   0.4770   
8890 2026-07-13    WMT  0.0373 -0.0482  0.2491 -0.4499       -0.7543   0.2210   
8891 2026-07-13    XOM  0.0591 -0.0406  0.3080  0.1651        0.2039  -0.4111   

      drawdown_63  target  
8889      -0.0606       0  
8890      -0.1891       0  
8891      -0.1630       1  
{
  "model_table_rows": 7896,
  "pipeline_topological_order": [
    "raw_prices",
    "returns",
    "labels",
    "rolling_features",
    "risk_features",
    "model_table",
    "walk_forward_model",
    "portfolio"
  ]
}


### Four classic standalone strategies (the algorithmic-backtester layer)

Before the ML model, every quant should be able to build and defend the four
textbook strategies. Each is the same loop: **signal -> daily weights ->
net-of-cost returns -> performance + risk**. These are deliberately separate
from the ML feature `mom_21`: here momentum is a *strategy*, not a column.

- **Momentum** - long 126-day winners, short losers (cross-sectional, monthly).
- **Mean reversion** - long 5-day losers, short winners (weekly).
- **MA crossover** - per stock, long when the 20-day MA tops the 50-day MA (trend).
- **Pairs trading** - the most-correlated pair, trade the 63-day z-scored spread.


In [6]:
# ---------------------------------------------------------------------------
# Four classic standalone strategies (distinct from the ML model in S3).
# Each: signal -> daily weights -> net-of-cost returns -> performance + risk.
# ---------------------------------------------------------------------------
px = DATA["prices"]
rx = DATA["returns"]
dates_idx = rx.index
universe = list(px.columns)

def strategy_stats(net_ret):
    clean = net_ret.dropna()
    eq = (1 + net_ret.fillna(0.0)).cumprod()  # aligned to the full date axis
    ann_ret = float(clean.mean() * 252)
    ann_vol = float(clean.std() * np.sqrt(252))
    return {
        "ann_return": ann_ret,
        "ann_vol": ann_vol,
        "sharpe": float(ann_ret / ann_vol) if ann_vol > 0 else 0.0,
        "max_drawdown": max_drawdown(eq),
        "hit_rate": float((clean > 0).mean()),
        "equity_curve": eq.round(5).tolist(),
    }

def backtest_weights(weights, cost_bps=5):
    weights = weights.reindex(dates_idx).fillna(0.0)
    gross = (weights.shift(1).fillna(0.0) * rx).sum(axis=1)
    turnover = weights.diff().abs().sum(axis=1).fillna(0.0)
    stats = strategy_stats(gross - turnover * cost_bps / 10000.0)
    stats["avg_turnover"] = float(turnover.mean())
    return stats

def rebalanced_long_short(signal, long_k=3, short_k=3, step=21):
    rebal = set(dates_idx[::step])
    w = pd.DataFrame(0.0, index=dates_idx, columns=universe)
    cur = pd.Series(0.0, index=universe)
    for d in dates_idx:
        if d in rebal and d in signal.index:
            cur = pd.Series(0.0, index=universe)
            r = signal.loc[d].dropna()
            if len(r) >= long_k + short_k:
                cur[r.nlargest(long_k).index] = 1.0 / long_k
                cur[r.nsmallest(short_k).index] = -1.0 / short_k
        w.loc[d] = cur
    return w

# 1) Momentum: 126-day return, long winners / short losers, monthly rebalance.
momentum_w = rebalanced_long_short(px.pct_change(126), step=21)
# 2) Mean reversion: 5-day reversal, long recent losers / short winners, weekly.
meanrev_w = rebalanced_long_short(-px.pct_change(5), step=5)
# 3) MA crossover: per-stock trend, long when fast(20) MA tops slow(50) MA.
ma_fast, ma_slow = px.rolling(20).mean(), px.rolling(50).mean()
crossover_w = np.sign(ma_fast - ma_slow) / len(universe)
# 4) Pairs trading: most-correlated pair, trade the 63-day z-scored log-spread.
corr_pairs = rx.corr()
np.fill_diagonal(corr_pairs.values, 0.0)
pair_a, pair_b = corr_pairs.stack().idxmax()
spread = np.log(px[pair_a]) - np.log(px[pair_b])
zspread = (spread - spread.rolling(63).mean()) / spread.rolling(63).std()
pair_pos = (-np.sign(zspread)).where(zspread.abs() > 1.0, 0.0).fillna(0.0)
pairs_w = pd.DataFrame(0.0, index=dates_idx, columns=universe)
pairs_w[pair_a] = 0.5 * pair_pos
pairs_w[pair_b] = -0.5 * pair_pos

strategy_weights = {
    "momentum": momentum_w,
    "mean_reversion": meanrev_w,
    "ma_crossover": crossover_w,
    "pairs_trading": pairs_w,
}
strategies = {name: backtest_weights(w) for name, w in strategy_weights.items()}

s2_strategies = {
    "laiqms_modules": ["S", "Q"],
    "dates": [str(d.date()) for d in dates_idx],
    "pairs_pair": [pair_a, pair_b],
    "strategies": strategies,
}
(ARTIFACTS / "s2_strategies.json").write_text(json.dumps(s2_strategies, indent=2))
print(pd.DataFrame({k: {m: v[m] for m in ["ann_return", "ann_vol", "sharpe", "max_drawdown", "hit_rate", "avg_turnover"]} for k, v in strategies.items()}).T.round(4))


                ann_return  ann_vol  sharpe  max_drawdown  hit_rate  \
momentum           -0.0135   0.2343 -0.0574       -0.3544    0.4381   
mean_reversion     -0.0715   0.2295 -0.3114       -0.4091    0.5087   
ma_crossover        0.0229   0.1106  0.2072       -0.1971    0.4807   
pairs_trading       0.0087   0.0692  0.1262       -0.1456    0.2450   

                avg_turnover  
momentum              0.0506  
mean_reversion        0.5974  
ma_crossover          0.0388  
pairs_trading         0.1478  


# S3 - ML + AI Model Validation

Train an honest walk-forward ensemble and produce a model card with leakage, calibration, AI evaluation, and deployment caveats.


## Mapped LAIQMS Subsections

- M: Time-Series Validation & Leakage Control
- M: Supervised Learning I: Bias, Variance & Splits
- M: Supervised Learning II: Models & Regularization
- M: Trees, Ensembles & Feature Selection
- M: Unsupervised Learning & Dimensionality Reduction
- M: Metrics, Model Selection & Imbalanced Data
- AI: Neural Network Foundations
- AI: Deep Learning Architectures
- AI: Representation Learning & Embeddings
- AI: Optimization, Regularization & Generalization
- AI: Applied AI Evaluation & Calibration
- AI: AI Systems, Fine-Tuning & Deployment


## S3 - ML + AI Model Validation

**Natural order:** after features, train the model. The goal is not fake accuracy; it is leak-free, defensible validation.

**LAIQMS mapping**
- M / Time-Series Validation: expanding walk-forward split and no future leakage.
- M / Bias-Variance: compare linear and nonlinear models.
- M / Models & Regularization: logistic regression, random forest, gradient boosting, MLP.
- M / Trees, Ensembles & Feature Selection: nonlinear baselines and feature importance.
- M / Unsupervised Learning: PCA/regime diagnostics before supervised claims.
- M / Metrics & Imbalanced Data: ROC-AUC, Brier, precision, recall, F1.
- AI / Neural Networks: small MLP classifier.
- AI / Deep Learning Architectures: feed-forward architecture and activation choices.
- AI / Representation Learning: latent feature and embedding view of market state.
- AI / Optimization & Generalization: regularization, early stopping, train/test curves.
- AI / Calibration: Brier score, probability bins, thresholds.
- AI / Deployment: model card, feature sensitivity, and monitoring gates.


In [7]:
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import roc_auc_score, brier_score_loss, precision_recall_fscore_support, accuracy_score

def expanding_splits(unique_dates, train_min=250, test_size=42, step=42):
    dates = list(pd.Index(unique_dates).sort_values())
    splits = []
    start = train_min
    while start + test_size <= len(dates):
        train_dates = dates[:start]
        test_dates = dates[start:start + test_size]
        splits.append((train_dates, test_dates))
        start += step
    return splits

def safe_auc(y, p):
    if len(np.unique(y)) < 2:
        return np.nan
    return float(roc_auc_score(y, p))

X = model_table[FEATURE_COLUMNS].astype(float)
y = model_table["target"].astype(int)
dates = pd.to_datetime(model_table["date"])

MODELS = {
    "logistic_l2": Pipeline([("scale", StandardScaler()), ("model", LogisticRegression(max_iter=1000, C=0.8))]),
    "random_forest": RandomForestClassifier(n_estimators=160, max_depth=5, min_samples_leaf=25, random_state=RNG_SEED, n_jobs=-1),
    "gradient_boosting": GradientBoostingClassifier(random_state=RNG_SEED, learning_rate=0.04, n_estimators=130, max_depth=2),
    "mlp": Pipeline([("scale", StandardScaler()), ("model", MLPClassifier(hidden_layer_sizes=(24, 12), alpha=0.02, max_iter=500, random_state=RNG_SEED))]),
}

preds = []
splits = expanding_splits(sorted(dates.unique()), train_min=252, test_size=42, step=42)
for fold, (train_dates, test_dates) in enumerate(splits, 1):
    train_mask = dates.isin(train_dates)
    test_mask = dates.isin(test_dates)
    fold_frame = model_table.loc[test_mask, ["date", "ticker", "target"]].copy()
    for name, model in MODELS.items():
        model.fit(X.loc[train_mask], y.loc[train_mask])
        if hasattr(model, "predict_proba"):
            fold_frame[name] = model.predict_proba(X.loc[test_mask])[:, 1]
        else:
            fold_frame[name] = model.decision_function(X.loc[test_mask])
    fold_frame["ensemble"] = fold_frame[list(MODELS)].mean(axis=1)
    fold_frame["fold"] = fold
    preds.append(fold_frame)

predictions = pd.concat(preds, ignore_index=True)
metrics = {}
for name in list(MODELS) + ["ensemble"]:
    p = predictions[name].astype(float)
    y_true = predictions["target"].astype(int)
    y_hat = (p >= p.median()).astype(int)
    pr, rc, f1, _ = precision_recall_fscore_support(y_true, y_hat, average="binary", zero_division=0)
    metrics[name] = {
        "roc_auc": safe_auc(y_true, p),
        "brier": float(brier_score_loss(y_true, np.clip(p, 1e-6, 1 - 1e-6))),
        "accuracy_median_threshold": float(accuracy_score(y_true, y_hat)),
        "precision": float(pr),
        "recall": float(rc),
        "f1": float(f1),
    }

# Permutation-style importance on the final training window for the tree model.
final_train = dates < sorted(dates.unique())[-84]
final_test = ~final_train
gbm = MODELS["gradient_boosting"]
gbm.fit(X.loc[final_train], y.loc[final_train])
base = safe_auc(y.loc[final_test], gbm.predict_proba(X.loc[final_test])[:, 1])
importance = {}
for col in FEATURE_COLUMNS:
    Xp = X.loc[final_test].copy()
    Xp[col] = rng.permutation(Xp[col].values)
    importance[col] = float(base - safe_auc(y.loc[final_test], gbm.predict_proba(Xp)[:, 1]))

model_card = {
    "laiqms_modules": ["M", "AI", "Q"],
    "splits": len(splits),
    "features": FEATURE_COLUMNS,
    "metrics": metrics,
    "permutation_importance_auc_drop": importance,
    "leakage_controls": [
        "labels use future returns only after the feature date",
        "walk-forward split uses past dates only",
        "scalers are fit inside each training fold",
        "reported metrics are out-of-sample by fold",
    ],
}
(ARTIFACTS / "s3_model_card.json").write_text(json.dumps(model_card, indent=2))
predictions.to_csv(ARTIFACTS / "s3_walk_forward_predictions.csv", index=False)
print(pd.DataFrame(metrics).T.round(4))


/private/tmp/rv2/lib/python3.10/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(


/private/tmp/rv2/lib/python3.10/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(


                   roc_auc   brier  accuracy_median_threshold  precision  \
logistic_l2         0.5239  0.2502                     0.5265     0.5265   
random_forest       0.5251  0.2508                     0.5146     0.5146   
gradient_boosting   0.5278  0.2542                     0.5236     0.5236   
mlp                 0.5107  0.2903                     0.5101     0.5101   
ensemble            0.5227  0.2532                     0.5172     0.5172   

                   recall      f1  
logistic_l2        0.5265  0.5265  
random_forest      0.5146  0.5146  
gradient_boosting  0.5238  0.5237  
mlp                0.5101  0.5101  
ensemble           0.5172  0.5172  


### Credit-risk model: Probability of Default, ratings, and Expected Loss

The same supervised toolkit (logistic regression + trees) powers the other
classic quant project: **credit risk**. We build a synthetic loan book where a
latent default propensity depends on debt-to-income, utilization, delinquencies,
employment, and income; then we learn it and turn predictions into the numbers a
risk desk actually uses:

- **PD** (Probability of Default) from a calibrated classifier.
- **Credit ratings** by mapping PD into AAA -> CCC bands.
- **Expected Loss** = PD x LGD x EAD (loss-given-default x exposure-at-default).
- **Risk categories** (Low/Medium/High) and **validation** (ROC-AUC, KS, Brier).


In [8]:
# ---------------------------------------------------------------------------
# Credit-risk model: synthetic loan book -> PD -> rating -> Expected Loss.
# ---------------------------------------------------------------------------
from scipy.stats import ks_2samp
from sklearn.model_selection import train_test_split

n_borrowers = 6000
credit = pd.DataFrame({
    "age": rng.integers(21, 70, n_borrowers).astype(float),
    "income": rng.lognormal(11.0, 0.5, n_borrowers),
    "loan_amount": rng.lognormal(9.8, 0.6, n_borrowers),
    "employment_years": rng.integers(0, 35, n_borrowers).astype(float),
    "num_delinquencies": rng.poisson(0.4, n_borrowers).astype(float),
    "credit_utilization": np.clip(rng.beta(2, 5, n_borrowers), 0, 1),
})
credit["debt_to_income"] = (credit["loan_amount"] / credit["income"]).clip(0, 3)

def zscore_col(s):
    return (s - s.mean()) / s.std()

# Latent log-odds of default: higher DTI/utilization/delinquencies raise it,
# more employment/income lower it. Sigmoid -> true PD -> Bernoulli default flag.
logit = (
    -2.2
    + 1.10 * zscore_col(credit["debt_to_income"])
    + 1.30 * zscore_col(credit["credit_utilization"])
    + 0.80 * zscore_col(credit["num_delinquencies"])
    - 0.50 * zscore_col(credit["employment_years"])
    - 0.40 * zscore_col(credit["income"])
)
pd_true = 1.0 / (1.0 + np.exp(-logit))
credit["default"] = (rng.random(n_borrowers) < pd_true).astype(int)

feat_cols = ["age", "income", "loan_amount", "employment_years", "num_delinquencies", "credit_utilization", "debt_to_income"]
Xc = credit[feat_cols].astype(float)
yc = credit["default"].astype(int)
Xtr, Xte, ytr, yte = train_test_split(Xc, yc, test_size=0.3, random_state=RNG_SEED, stratify=yc)

credit_models = {
    "logistic": Pipeline([("scale", StandardScaler()), ("model", LogisticRegression(max_iter=1000, C=1.0))]),
    "random_forest": RandomForestClassifier(n_estimators=200, max_depth=6, min_samples_leaf=20, random_state=RNG_SEED, n_jobs=-1),
}
credit_metrics, pd_test = {}, None
for name, model in credit_models.items():
    model.fit(Xtr, ytr)
    p = model.predict_proba(Xte)[:, 1]
    credit_metrics[name] = {
        "roc_auc": float(roc_auc_score(yte, p)),
        "ks_statistic": float(ks_2samp(p[yte.values == 1], p[yte.values == 0]).statistic),
        "brier": float(brier_score_loss(yte, p)),
    }
    if name == "logistic":
        pd_test = p

RATING_BANDS = [(0.01, "AAA"), (0.02, "AA"), (0.05, "A"), (0.10, "BBB"), (0.20, "BB"), (0.40, "B")]
def rating_for_pd(pd_val):
    for thr, label in RATING_BANDS:
        if pd_val < thr:
            return label
    return "CCC"

LGD = 0.45  # loss given default (structural assumption, documented)
ead = Xte["loan_amount"].values  # exposure at default
expected_loss = pd_test * LGD * ead
ratings = pd.Series([rating_for_pd(p) for p in pd_test])
rating_order = ["AAA", "AA", "A", "BBB", "BB", "B", "CCC"]
risk_category = pd.cut(pd_test, bins=[-0.001, 0.05, 0.15, 1.0], labels=["Low", "Medium", "High"])

credit_report = {
    "laiqms_modules": ["M", "AI"],
    "n_borrowers": int(n_borrowers),
    "base_default_rate": float(yc.mean()),
    "metrics": credit_metrics,
    "rating_distribution": ratings.value_counts().reindex(rating_order).fillna(0).astype(int).to_dict(),
    "risk_category_distribution": pd.Series(risk_category).value_counts().reindex(["Low", "Medium", "High"]).fillna(0).astype(int).to_dict(),
    "lgd_assumption": LGD,
    "portfolio_ead": float(ead.sum()),
    "portfolio_expected_loss": float(expected_loss.sum()),
    "expected_loss_rate": float(expected_loss.sum() / ead.sum()),
    "pd_histogram": {"counts": np.histogram(pd_test, bins=20, range=(0, 1))[0].tolist(), "bin_width": 0.05},
    "feature_importance": dict(zip(feat_cols, np.round(credit_models["random_forest"].feature_importances_, 4).tolist())),
}
(ARTIFACTS / "s3_credit_risk.json").write_text(json.dumps(credit_report, indent=2))
print(json.dumps({k: credit_report[k] for k in ["base_default_rate", "metrics", "rating_distribution", "expected_loss_rate"]}, indent=2))


{
  "base_default_rate": 0.20766666666666667,
  "metrics": {
    "logistic": {
      "roc_auc": 0.8733415334768357,
      "ks_statistic": 0.5909278412372216,
      "brier": 0.10633927613291488
    },
    "random_forest": {
      "roc_auc": 0.8655563972369518,
      "ks_statistic": 0.5722525144190024,
      "brier": 0.11170878377693243
    }
  },
  "rating_distribution": {
    "AAA": 151,
    "AA": 192,
    "A": 330,
    "BBB": 284,
    "BB": 273,
    "B": 256,
    "CCC": 314
  },
  "expected_loss_rate": 0.11656214789684025
}


# S4 - LLM Research Layer

Build a local RAG-style research assistant with tokenization, embeddings, attention math, Transformer mechanics, inference, and grounded finance checks.


## Mapped LAIQMS Subsections

- L: Tokenization, Embeddings & Vector Geometry
- L: Softmax, Entropy & Cross-Entropy
- L: Attention, Masking & Multi-Head Geometry
- L: Transformer Blocks & Positional Encoding
- L: Backpropagation & Optimization
- L: Scaling Laws, FLOPs & Memory
- L: Evaluation, Calibration & Information Theory
- L: RAG, Inference & Alignment Workflows


## S4 - LLM Research Layer

**Natural order:** after the model exists, use LLM workflow for research, explanation, and monitoring. The LLM proposes; quant tests verify.

**LAIQMS mapping**
- L / Tokenization: simple financial-token parser.
- L / Embeddings: TF-IDF vector geometry and cosine retrieval.
- L / Softmax/Cross-Entropy: stable softmax and entropy for retrieval confidence.
- L / Attention: scaled dot-product attention from scratch.
- L / Transformer Blocks: residual stream, position information, attention block, and MLP block.
- L / Backpropagation & Optimization: gradient path through a retrieval scorer.
- L / Scaling Laws/FLOPs/Memory: token, context, prefill/decode, and serving-budget math.
- L / Evaluation/Calibration/Information Theory: likelihood, entropy, KL intuition, confidence.
- L / RAG: retrieve evidence before writing the research brief.


In [9]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

def stable_softmax(x):
    x = np.asarray(x, dtype=float)
    z = x - np.max(x)
    e = np.exp(z)
    return e / e.sum()

def entropy(p):
    p = np.clip(np.asarray(p, dtype=float), 1e-12, 1)
    return float(-(p * np.log(p)).sum())

def scaled_dot_product_attention(Q, K, V, mask=None):
    scores = Q @ K.T / np.sqrt(K.shape[-1])
    if mask is not None:
        scores = np.where(mask, scores, -1e9)
    weights = np.apply_along_axis(stable_softmax, 1, scores)
    return weights @ V, weights

docs = []
for t in TICKERS:
    last_mom = float(mom_21[t].dropna().iloc[-1])
    last_vol = float(vol_21[t].dropna().iloc[-1])
    last_beta = float(beta_63[t].dropna().iloc[-1])
    docs.append({
        "ticker": t,
        "text": (
            f"{t} latest research note: 21-day momentum {last_mom:.3f}; "
            f"annualized volatility {last_vol:.3f}; beta {last_beta:.3f}; "
            "risk checks include liquidity, sector exposure, drawdown, and transaction costs."
        ),
    })
docs += [
    {"ticker": "SYSTEM", "text": "Point-in-time data is mandatory. Avoid look-ahead bias, survivorship bias, and restated fundamentals."},
    {"ticker": "SYSTEM", "text": "A signal must survive walk-forward validation, transaction costs, turnover constraints, and stress periods."},
    {"ticker": "SYSTEM", "text": "LLM output is useful for parsing and hypothesis generation, but claims must cite retrieved evidence and numerical tests."},
]

vectorizer = TfidfVectorizer(ngram_range=(1, 2), min_df=1)
Xdoc = vectorizer.fit_transform([d["text"] for d in docs])

def retrieve(query, k=4):
    q = vectorizer.transform([query])
    sims = cosine_similarity(q, Xdoc).ravel()
    idx = np.argsort(-sims)[:k]
    probs = stable_softmax(sims[idx])
    return [{"score": float(sims[i]), "confidence": float(probs[j]), **docs[i]} for j, i in enumerate(idx)]

query = "Find candidates with momentum, controlled volatility, beta risk, and leak-free validation."
retrieved = retrieve(query, 5)

# Attention demo over retrieved document vectors.
dense = Xdoc[np.argsort(-cosine_similarity(vectorizer.transform([query]), Xdoc).ravel())[:5]].toarray()
Q = dense[:1]
K = dense
V = dense
attended, attn_weights = scaled_dot_product_attention(Q, K, V)

def grounded_brief(ticker):
    evidence = retrieve(f"{ticker} momentum volatility beta risk validation", 4)
    pred_slice = predictions[predictions["ticker"] == ticker].copy()
    score = float(pred_slice["ensemble"].tail(30).mean()) if len(pred_slice) else np.nan
    risk = {
        "vol": float(vol_21[ticker].dropna().iloc[-1]),
        "beta": float(beta_63[ticker].dropna().iloc[-1]),
        "drawdown": float(drawdown_63[ticker].dropna().iloc[-1]),
    }
    unsupported_claims = []
    if score > 0.65 and risk["vol"] > vol_21.stack().quantile(0.85):
        unsupported_claims.append("High model score conflicts with high volatility; require position cap.")
    return {
        "ticker": ticker,
        "model_score_30d_mean": score,
        "risk": risk,
        "evidence": evidence,
        "decision": "candidate" if score >= 0.52 and risk["vol"] < vol_21.stack().quantile(0.75) else "watchlist",
        "unsupported_claims": unsupported_claims,
    }

briefs = [grounded_brief(t) for t in TICKERS]
rag_report = {
    "laiqms_modules": ["L", "AI", "Q"],
    "query": query,
    "retrieval_entropy": entropy([r["confidence"] for r in retrieved]),
    "attention_weights": attn_weights.round(4).tolist(),
    "top_retrieval": retrieved,
    "briefs": briefs,
}
(ARTIFACTS / "s4_rag_research_briefs.json").write_text(json.dumps(rag_report, indent=2))
print(json.dumps({"retrieval_entropy": rag_report["retrieval_entropy"], "top": retrieved[0]}, indent=2))


{
  "retrieval_entropy": 1.6092659380795997,
  "top": {
    "score": 0.18274932820901232,
    "confidence": 0.20745244594493362,
    "ticker": "SYSTEM",
    "text": "A signal must survive walk-forward validation, transaction costs, turnover constraints, and stress periods."
  }
}


# S5 - Portfolio, Execution, Risk

Turn model scores into a constrained portfolio, include Greeks, costs, stress tests, execution controls, and production risk gates.


## Mapped LAIQMS Subsections

- Q: Portfolio Theory, Factors & Risk
- Q: Financial Instruments & Derivatives
- Q: Black–Scholes
- Q: Monte Carlo & Numerical Methods
- M: Metrics, Model Selection & Imbalanced Data
- S: Stream Processing, Queues & Backpressure
- S: Applied Trading and ML System Design


## S5 - Portfolio, Execution, Risk

**Natural order:** a model is not a strategy until position sizing, transaction costs, risk limits, and execution logic exist.

**LAIQMS mapping**
- Q / Portfolio Theory: optimizer, constraints, Sharpe, drawdown, beta, VaR, expected shortfall.
- Q / Instruments: turnover, costs, instrument choice, option overlay, and product mechanics.
- Q / Black–Scholes: option overlay, delta/gamma/vega exposure, and hedge ratio.
- Q / Monte Carlo: stress scenarios, VaR, expected shortfall, drawdown.
- M / Metrics: model score thresholding and out-of-sample interpretation.
- S / Streams/Queues: rebalance queue and order throttling.
- S / Trading System Design: market-data -> signal -> risk -> order path.


In [10]:
def normalize_long_short(scores, long_k=3, short_k=3, gross=1.0):
    scores = scores.dropna().sort_values()
    shorts = scores.head(short_k).index
    longs = scores.tail(long_k).index
    w = pd.Series(0.0, index=scores.index)
    if len(longs):
        w.loc[longs] = gross / 2 / len(longs)
    if len(shorts):
        w.loc[shorts] = -gross / 2 / len(shorts)
    return w

pred_wide = predictions.pivot_table(index="date", columns="ticker", values="ensemble", aggfunc="last").sort_index()
ret = DATA["returns"].reindex(pred_wide.index)
rebal_dates = pred_wide.index[::21]
weights_by_day = pd.DataFrame(0.0, index=pred_wide.index, columns=TICKERS)
current = pd.Series(0.0, index=TICKERS)
rebalance_queue = deque()

for d in pred_wide.index:
    if d in set(rebal_dates):
        target = normalize_long_short(pred_wide.loc[d], 3, 3, gross=1.0).reindex(TICKERS).fillna(0.0)
        rebalance_queue.append({"date": str(d.date()), "orders": (target - current).round(5).to_dict()})
        current = target
    weights_by_day.loc[d] = current

strategy_ret_gross = (weights_by_day.shift().fillna(0.0) * ret).sum(axis=1)
turnover = weights_by_day.diff().abs().sum(axis=1).fillna(0.0)
cost_bps = 6
strategy_ret = strategy_ret_gross - turnover * cost_bps / 10000
wealth = (1 + strategy_ret).cumprod()
bench = ret.mean(axis=1).reindex(strategy_ret.index)

beta = float(np.cov(strategy_ret.dropna(), bench.loc[strategy_ret.dropna().index].dropna())[0, 1] / np.var(bench.dropna()))
alpha_daily = float(strategy_ret.mean() - beta * bench.mean())
stress = {}
for name, shock in {"mild_liquidity_cost": 0.0005, "severe_liquidity_cost": 0.0015, "crisis_gap": 0.006}.items():
    stressed = strategy_ret - turnover * shock
    if name == "crisis_gap":
        stressed = stressed - (DATA["regime"].reindex(stressed.index).fillna(0).eq(2).astype(float) * shock)
    stress[name] = {
        "ann_return": float(stressed.mean() * 252),
        "ann_vol": float(stressed.std() * np.sqrt(252)),
        "max_drawdown": max_drawdown((1 + stressed).cumprod()),
    }

# Simple option overlay example: delta hedge one long call exposure on the top candidate.
latest_scores = pred_wide.iloc[-1].sort_values()
top_name = latest_scores.index[-1]
S0 = float(prices[top_name].iloc[-1])
sig = float(vol_21[top_name].dropna().iloc[-1])
call = black_scholes_call(S0, S0 * 1.03, 45 / 365, RISK_FREE, sig)
delta_hedge_shares = -call["delta"]

risk_report = {
    "laiqms_modules": ["Q", "M", "S"],
    "ann_return": float(strategy_ret.mean() * 252),
    "ann_vol": float(strategy_ret.std() * np.sqrt(252)),
    "sharpe": float((strategy_ret.mean() * 252 - RISK_FREE) / (strategy_ret.std() * np.sqrt(252))),
    "max_drawdown": max_drawdown(wealth),
    "beta_to_equal_weight": beta,
    "alpha_daily": alpha_daily,
    "daily_var_95": float(np.quantile(strategy_ret.dropna(), 0.05)),
    "daily_expected_shortfall_95": float(strategy_ret[strategy_ret <= np.quantile(strategy_ret.dropna(), 0.05)].mean()),
    "average_turnover": float(turnover.mean()),
    "option_overlay": {"ticker": top_name, "call": call, "delta_hedge_shares_per_call": float(delta_hedge_shares)},
    "stress": stress,
    "latest_rebalance_order": rebalance_queue[-1] if rebalance_queue else {},
    "execution_controls": [
        "position cap by volatility bucket",
        "turnover throttle",
        "liquidity/cost gate before order creation",
        "model-score stale check",
        "kill switch when drawdown or data freshness breaches limits",
    ],
}
(ARTIFACTS / "s5_strategy_risk_report.json").write_text(json.dumps(risk_report, indent=2))
print(pd.Series({k: risk_report[k] for k in ["ann_return", "ann_vol", "sharpe", "max_drawdown", "average_turnover"]}).round(4))


ann_return          0.1069
ann_vol             0.1153
sharpe              0.5543
max_drawdown       -0.1075
average_turnover    0.0547
dtype: float64


### Portfolio optimization, multi-asset frontier, and the option engine

The final quant-project pieces: the **efficient frontier** (the classic
risk/return parabola), a **multi-asset allocation** across Equities / Bonds /
Commodities / Cash, an **implied-volatility surface** (skew + term structure),
and a **multi-leg option strategy simulator** (straddle, spread, protective put).


In [11]:
# ---------------------------------------------------------------------------
# Efficient frontier (closed-form Lagrange), multi-asset max-Sharpe allocation,
# implied-vol surface, and multi-leg option payoff diagrams.
# ---------------------------------------------------------------------------
from scipy.optimize import minimize

def frontier_points(mu_vec, cov, n_points=25):
    # Min-variance portfolio for each target return (allows shorting): closed form.
    inv = np.linalg.pinv(cov)
    ones = np.ones(len(mu_vec))
    a = float(ones @ inv @ ones)
    b = float(ones @ inv @ mu_vec)
    c = float(mu_vec @ inv @ mu_vec)
    det = a * c - b * b
    pts = []
    for tgt in np.linspace(mu_vec.min(), mu_vec.max(), n_points):
        lam = (c - b * tgt) / det
        gam = (a * tgt - b) / det
        w = inv @ (lam * ones + gam * mu_vec)
        pts.append({"target_return": float(tgt), "vol": float(np.sqrt(max(w @ cov @ w, 0.0)))})
    return pts

def max_sharpe_constrained(mu_vec, cov, rf, w_min=0.05, w_max=0.50):
    # Long-only max-Sharpe with per-asset position limits (a real-world constraint
    # that also keeps the allocation diversified instead of cornering on one asset).
    nA = len(mu_vec)
    def neg_sharpe(w):
        pr, pv = w @ mu_vec, np.sqrt(w @ cov @ w)
        return -(pr - rf) / pv if pv > 0 else 0.0
    res = minimize(neg_sharpe, np.repeat(1.0 / nA, nA), method="SLSQP",
                   bounds=[(w_min, w_max)] * nA, constraints=[{"type": "eq", "fun": lambda w: w.sum() - 1.0}])
    return res.x

# Equity efficient frontier.
eq_ret = DATA["returns"].dropna()
eq_mu, eq_cov = (eq_ret.mean() * 252).values, (eq_ret.cov() * 252).values
equity_frontier = frontier_points(eq_mu, eq_cov)

# Multi-asset frontier + long-only max-Sharpe allocation pie.
ac = DATA["asset_classes"].dropna()
ac_mu, ac_cov = (ac.mean() * 252).values, (ac.cov() * 252).values
ac_frontier = frontier_points(ac_mu, ac_cov)
ac_weights = max_sharpe_constrained(ac_mu, ac_cov, RISK_FREE)
allocation = {cls: float(round(w, 4)) for cls, w in zip(ac.columns, ac_weights)}

# Implied-vol surface: synthesize skew + term structure, price with BS, invert to IV.
spot = float(DATA["prices"].iloc[-1].mean())
base_vol = 0.22
moneyness = np.round(np.linspace(0.85, 1.15, 13), 3)
maturities_days = [30, 60, 90, 180, 365]
vol_surface = []
for days in maturities_days:
    T = days / 365.0
    row = []
    for m in moneyness:
        K = spot * m
        iv_in = base_vol + 0.10 * (1.0 - m) + 0.03 * np.sqrt(T)  # skew + term effect
        price = black_scholes_call(spot, K, T, RISK_FREE, iv_in)["price"]
        row.append(round(float(implied_vol_call(price, spot, K, T, RISK_FREE)), 4))
    vol_surface.append(row)

# Multi-leg option strategies: net P&L at expiry (gross payoff minus net premium).
T0 = 30 / 365.0
def call_px(K): return black_scholes_call(spot, K, T0, RISK_FREE, base_vol)["price"]
def put_px(K): return black_scholes_call(spot, K, T0, RISK_FREE, base_vol)["price"] - spot + K * np.exp(-RISK_FREE * T0)  # put-call parity
S_grid = np.round(np.linspace(spot * 0.7, spot * 1.3, 61), 2)
co = lambda K: np.maximum(S_grid - K, 0.0)
po = lambda K: np.maximum(K - S_grid, 0.0)
payoffs = {
    "long_straddle": (co(spot) + po(spot) - call_px(spot) - put_px(spot)),
    "bull_call_spread": (co(spot) - co(spot * 1.1) - call_px(spot) + call_px(spot * 1.1)),
    "protective_put": (S_grid - spot) + po(spot * 0.95) - put_px(spot * 0.95),
}
payoffs = {k: np.round(v, 3).tolist() for k, v in payoffs.items()}

portfolio_analytics = {
    "laiqms_modules": ["Q", "M", "S"],
    "equity_frontier": equity_frontier,
    "asset_class_frontier": ac_frontier,
    "asset_class_allocation": allocation,
    "vol_surface": {"moneyness": moneyness.tolist(), "maturities_days": maturities_days, "iv": vol_surface},
    "option_payoffs": {"spot": spot, "underlying_grid": S_grid.tolist(), "strategies": payoffs},
}
(ARTIFACTS / "s5_portfolio_analytics.json").write_text(json.dumps(portfolio_analytics, indent=2))
print("Multi-asset max-Sharpe allocation:", allocation)
print("Vol surface (30d row):", vol_surface[0])


Multi-asset max-Sharpe allocation: {'Equities': 0.5, 'Bonds': 0.05, 'Commodities': 0.3134, 'Cash': 0.1366}
Vol surface (30d row): [0.2436, 0.2411, 0.2386, 0.2361, 0.2336, 0.2311, 0.2286, 0.2261, 0.2236, 0.2211, 0.2186, 0.2161, 0.2136]


# S6 - AWS, System Design, Final Defense

Package the capstone as a production architecture with AWS services, monitoring, security, cost controls, and interview-defense prompts.


## Mapped LAIQMS Subsections

- S: System Design Fundamentals & Estimation
- S: Online Processing Systems
- S: Batch Processing & Storage Systems
- S: Stream Processing, Queues & Backpressure
- S: Distributed Consistency & Fault Tolerance
- S: Applied Trading and ML System Design
- S: AWS Cloud Concepts & Economics
- S: AWS Shared Responsibility, IAM & Security
- S: AWS Compute & Serverless
- S: AWS Storage & Databases
- S: AWS Networking & Content Delivery
- S: AWS Monitoring, Reliability & Operations
- S: AWS Billing, Pricing & Support


## S6 - AWS, System Design, Final Defense

**Natural order:** once the strategy works as a notebook, design the production path.

**LAIQMS mapping**
- S / System Design: ingestion, batch jobs, online APIs, queues, failure modes.
- AWS / Cloud Practitioner: IAM, S3, Lambda, ECS, DynamoDB, CloudWatch, EventBridge, VPC, CloudFront, budgets.
- L / AI / M / Q: the deployed system must monitor model drift, LLM grounding, data freshness, and portfolio risk.


In [12]:
architecture = {
    "name": "laiqms-alpha-research-and-risk-system",
    "data_layer": {
        "raw_market_data": "s3://mafn-capstone/raw/market/",
        "feature_store": "s3://mafn-capstone/curated/features/",
        "model_registry": "s3://mafn-capstone/models/",
        "research_corpus": "s3://mafn-capstone/rag/documents/",
    },
    "compute": {
        "batch_feature_job": "AWS Batch or ECS scheduled task",
        "model_training": "SageMaker training job or ECS task",
        "inference_api": "Lambda/API Gateway for low-volume scoring; ECS/Fargate for heavier use",
        "notebook": "SageMaker Studio or local Jupyter template",
    },
    "state": {
        "predictions": "DynamoDB table keyed by date#ticker",
        "risk_reports": "S3 JSON artifacts and Athena external table",
        "strategy_status": "S3 JSON strategy state (live / review-only / halted) behind authenticated API",
    },
    "networking": {
        "cdn": "CloudFront for research-artifact distribution",
        "private_subnets": "model jobs and databases",
        "public_edge": "API Gateway + WAF",
    },
    "security": {
        "iam": "least-privilege role per job",
        "encryption": "SSE-S3 or KMS for S3; DynamoDB encryption at rest",
        "auth": "identity-provider group membership gates promotion to live trading",
        "audit": "CloudTrail + application audit log",
    },
    "monitoring": {
        "data_freshness": "CloudWatch metric: minutes since last market-data update",
        "model_drift": "population stability index / feature distribution shift",
        "llm_grounding": "retrieval evidence count and unsupported-claim rate",
        "risk": "drawdown, VaR, turnover, beta, exposure caps",
    },
}

iam_policy_example = {
    "Version": "2012-10-17",
    "Statement": [
        {"Effect": "Allow", "Action": ["s3:GetObject", "s3:PutObject"], "Resource": ["arn:aws:s3:::mafn-capstone/*"]},
        {"Effect": "Allow", "Action": ["dynamodb:GetItem", "dynamodb:PutItem", "dynamodb:Query"], "Resource": ["arn:aws:dynamodb:us-east-1:123456789012:table/mafn-capstone-*"]},
        {"Effect": "Allow", "Action": ["cloudwatch:PutMetricData"], "Resource": "*"},
    ],
}

step_functions_skeleton = {
    "Comment": "LAIQMS capstone daily pipeline",
    "StartAt": "IngestMarketData",
    "States": {
        "IngestMarketData": {"Type": "Task", "Resource": "arn:aws:lambda:REGION:ACCT:function:ingest-market-data", "Next": "BuildFeatures"},
        "BuildFeatures": {"Type": "Task", "Resource": "arn:aws:states:::batch:submitJob.sync", "Next": "ScoreModel"},
        "ScoreModel": {"Type": "Task", "Resource": "arn:aws:lambda:REGION:ACCT:function:score-model", "Next": "RiskChecks"},
        "RiskChecks": {"Type": "Task", "Resource": "arn:aws:lambda:REGION:ACCT:function:risk-checks", "Next": "PublishReports"},
        "PublishReports": {"Type": "Task", "Resource": "arn:aws:lambda:REGION:ACCT:function:publish-reports", "End": True},
    },
}

monthly_cost_estimate = pd.DataFrame([
    {"service": "S3", "assumption": "50 GB artifacts/data", "monthly_usd": 1.25},
    {"service": "Lambda", "assumption": "100k requests", "monthly_usd": 2.00},
    {"service": "DynamoDB", "assumption": "on-demand prediction store", "monthly_usd": 5.00},
    {"service": "CloudWatch", "assumption": "metrics/logs/alarms", "monthly_usd": 8.00},
    {"service": "ECS/Batch", "assumption": "scheduled training jobs", "monthly_usd": 30.00},
    {"service": "CloudFront", "assumption": "notebook downloads", "monthly_usd": 3.00},
])

runbook = [
    "If data freshness > threshold: stop scoring, page the on-call researcher, publish a stale-data banner.",
    "If model drift breaches threshold: keep the incumbent model, require sign-off before promotion.",
    "If unsupported LLM claim rate rises: disable research-assistant output until grounding is restored.",
    "If drawdown or turnover limit breaches: set strategy status to review-only.",
    "If cost budget forecast exceeds limit: reduce training frequency and cap artifact retention.",
]

defense_prompts = [
    "Explain why the model is not allowed to train on future data.",
    "Defend the portfolio optimizer against noisy expected returns.",
    "Explain where the LLM is useful and where it must be constrained.",
    "Draw the AWS production architecture and name the failure modes.",
    "Describe the monitoring that would stop this strategy from going live.",
]

s6 = {
    "laiqms_modules": ["S", "AWS", "L", "AI", "Q", "M"],
    "architecture": architecture,
    "iam_policy_example": iam_policy_example,
    "step_functions_skeleton": step_functions_skeleton,
    "monthly_cost_estimate": monthly_cost_estimate.to_dict(orient="records"),
    "runbook": runbook,
    "defense_prompts": defense_prompts,
}
(ARTIFACTS / "s6_aws_architecture.json").write_text(json.dumps(s6, indent=2))
(ARTIFACTS / "s6_final_defense_prompts.md").write_text("\n".join(f"- {x}" for x in defense_prompts))
print(monthly_cost_estimate)


      service                  assumption  monthly_usd
0          S3        50 GB artifacts/data         1.25
1      Lambda               100k requests         2.00
2    DynamoDB  on-demand prediction store         5.00
3  CloudWatch         metrics/logs/alarms         8.00
4   ECS/Batch     scheduled training jobs        30.00
5  CloudFront          notebook downloads         3.00


### SQL analytics layer (the data-querying skill)

Every quant role expects fluent SQL. Using stdlib **sqlite3** (no server, runs
in-browser too), we load the model predictions, the strategy leaderboard, and the
credit loan book into tables and answer the questions a desk actually asks:
*top candidates*, *best strategy by Sharpe*, *expected loss by rating*.


In [13]:
# ---------------------------------------------------------------------------
# SQL analytics over the capstone outputs using stdlib sqlite3 (in-memory).
# ---------------------------------------------------------------------------
import sqlite3
conn = sqlite3.connect(":memory:")

# 1) Model predictions -> top candidates by average ensemble score.
predictions.to_sql("predictions", conn, index=False, if_exists="replace")
top_candidates = pd.read_sql_query(
    "SELECT ticker, ROUND(AVG(ensemble), 4) AS avg_score, COUNT(*) AS n_obs "
    "FROM predictions GROUP BY ticker ORDER BY avg_score DESC LIMIT 5", conn)

# 2) Strategy leaderboard -> best strategy by Sharpe.
strat_df = pd.DataFrame(
    {k: {m: v[m] for m in ["ann_return", "ann_vol", "sharpe", "max_drawdown"]} for k, v in strategies.items()}
).T.reset_index().rename(columns={"index": "strategy"})
strat_df.to_sql("strategies", conn, index=False, if_exists="replace")
best_strategy = pd.read_sql_query(
    "SELECT strategy, ROUND(sharpe, 4) AS sharpe FROM strategies ORDER BY sharpe DESC LIMIT 3", conn)

# 3) Credit loan book -> expected loss by rating.
loans = pd.DataFrame({"rating": ratings.values, "pd": pd_test, "expected_loss": expected_loss, "loan_amount": ead})
loans.to_sql("loans", conn, index=False, if_exists="replace")
el_by_rating = pd.read_sql_query(
    "SELECT rating, COUNT(*) AS n, ROUND(AVG(pd), 4) AS avg_pd, ROUND(SUM(expected_loss), 0) AS total_el "
    "FROM loans GROUP BY rating ORDER BY total_el DESC", conn)
conn.close()

s6_sql = {
    "laiqms_modules": ["S"],
    "engine": "sqlite3 (Python stdlib, in-memory)",
    "tables": ["predictions", "strategies", "loans"],
    "top_candidates": top_candidates.to_dict(orient="records"),
    "best_strategy": best_strategy.to_dict(orient="records"),
    "expected_loss_by_rating": el_by_rating.to_dict(orient="records"),
}
(ARTIFACTS / "s6_sql_analytics.json").write_text(json.dumps(s6_sql, indent=2))
print(top_candidates.to_string(index=False))
print(el_by_rating.to_string(index=False))


ticker  avg_score  n_obs
   UNH     0.5412    378
    GS     0.5378    378
   CAT     0.5337    378
    GE     0.5288    378
   JPM     0.5234    378
rating   n  avg_pd  total_el
   CCC 314  0.6578 3066426.0
     B 256  0.2907  796062.0
    BB 273  0.1465  365481.0
   BBB 284  0.0729  177244.0
     A 330  0.0328   95912.0
    AA 192  0.0147   21546.0
   AAA 151  0.0056    5539.0
